# 06. CoT vs 고객 DB 대표성 비교 (2026-11-17)

판정 없이 사실만 정리한다. 해석은 맨 아래 '06 정리 (직접 작성)' 칸에 직접 쓴다.

- **CoT 쪽**: 증권 CoT 전체(Training + Validation, 7,201건). 메타 `age`·`gender`, 성향은 D21(cot2 단일 성향 우선, 없으면 cot1 단일 성향).
- **고객 쪽**: `소비자.csv`(Training 폴더 파일 하나, D1)의 `customer_type == '증권'` 고객. 고객 단위로 중복 제거(`customer_id`).
- 둘 다 AI 학습용 합성데이터다. 실제 고객 행동으로 해석하지 않는다.
- 비율은 각 쪽 합계 대비 %, 차이는 CoT% − 고객% (퍼센트포인트, %p).

In [ ]:
import sys, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.load_cot import load_cot
from src.extract import propensity_in, extract_age_decade, PROPENSITY

pd.set_option("display.width", 200)
cot = load_cot()
sc = cot[cot["category"] == "증권"].copy()
print("증권 CoT:", len(sc), sc["split"].value_counts().to_dict())

In [ ]:
# 고객 DB: 필요한 컬럼만 읽는다
zp = ROOT / "data" / "raw" / "Training" / "02.라벨링데이터" / "TL_2.소비자.zip"
with zipfile.ZipFile(zp, metadata_encoding="cp949") as z:
    with z.open(z.namelist()[0]) as f:
        cons = pd.read_csv(f, usecols=["customer_id", "customer_type", "age", "gender", "investment_propensity"],
                           encoding="utf-8-sig")
cs_rows = cons[cons["customer_type"] == "증권"]
print("증권 행:", len(cs_rows), "/ 고객:", cs_rows["customer_id"].nunique())
# 한 고객 안에서 값이 여러 개인지 확인 (여러 개면 중복 제거 기준이 필요)
print("고객 내 값이 2개 이상인 고객 수:",
      {c: int((cs_rows.groupby("customer_id")[c].nunique() > 1).sum()) for c in ["age", "gender", "investment_propensity"]})
cs = cs_rows.drop_duplicates("customer_id").copy()
print("중복 제거 후 고객:", len(cs))

## 1. 연령대 표기 맞추기

- 아래에서 두 쪽의 연령대 값을 그대로 센다. 고객 DB 증권 고객에는 '70대 이상' 값이 없다(보험 고객에만 있다). 그래서 고객 쪽 연령대는 그대로 둔다.
- CoT 쪽의 '50대 이상'(2건), '60대 이상'(1건)은 D22대로 question 본문의 연령대로 맞춘다. 본문에서 연령대를 못 찾으면 '확인 불가'로 두고 비교에서 뺀다.
- 두 쪽 모두 '70대'와 '80대 이상'이 따로 있어 그대로 비교한다. 참고로 둘을 합친 '70대 이상' 행도 따로 출력한다.

In [ ]:
print("CoT age:", sc["age"].value_counts().to_dict())
print("고객 age:", cs["age"].value_counts().to_dict())

def fix_age(r):
    if r["age"] in ("50대 이상", "60대 이상"):
        d = extract_age_decade(r["question"])
        if d is None or pd.isna(d):
            return "확인 불가"
        d = int(d)  # 십의 자리 숫자 (예: 5 → 50대)
        return "80대 이상" if d >= 8 else f"{d * 10}대"
    return r["age"]

sc["연령대"] = sc.apply(fix_age, axis=1)
print("D22 교정:", sc.loc[sc["age"].isin(["50대 이상", "60대 이상"]), ["cot_id", "age", "연령대"]].to_string(index=False))
cs["연령대"] = cs["age"]
AGE_ORDER = ["20대", "30대", "40대", "50대", "60대", "70대", "80대 이상"]
print("비교에서 빠지는 값 — CoT:", sorted(set(sc["연령대"]) - set(AGE_ORDER)), "/ 고객:", sorted(set(cs["연령대"]) - set(AGE_ORDER)))

## 2. 성향 (D21)

- CoT: cot2에 성향이 하나만 있으면 그것, 아니면 cot1에 하나만 있을 때 그것. 둘 다 아니면 '비교 불가'로 두고 비교에서 뺀다.
- 고객: `investment_propensity`. 증권 고객에는 '-'가 없다(D4).

In [ ]:
sc["성향_cot1"] = sc["cot1"].map(propensity_in)
sc["성향_cot2"] = sc["cot2"].map(propensity_in)
sc["성향"] = np.where(sc["성향_cot2"].isin(PROPENSITY), sc["성향_cot2"],
                    np.where(sc["성향_cot1"].isin(PROPENSITY), sc["성향_cot1"], "비교 불가"))
print("CoT 성향:", sc["성향"].value_counts().to_dict())
cs["성향"] = cs["investment_propensity"]
print("고객 성향:", cs["성향"].value_counts().to_dict())
PROP_ORDER = ["안정형", "안정추구형", "위험중립형", "적극투자형", "공격투자형"]

## 3. 항목별 비율 표와 차이(%p)

In [ ]:
def compare(item, order, col):
    a = sc[sc[col].isin(order)][col].value_counts()
    b = cs[cs[col].isin(order)][col].value_counts()
    t = pd.DataFrame({"CoT 수": a, "고객 수": b}).reindex(order).fillna(0).astype(int)
    t["CoT(%)"] = (t["CoT 수"] / t["CoT 수"].sum() * 100).round(1)
    t["고객(%)"] = (t["고객 수"] / t["고객 수"].sum() * 100).round(1)
    t["차이(%p)"] = (t["CoT(%)"] - t["고객(%)"]).round(1)
    t["CoT/고객 배율"] = (t["CoT(%)"] / t["고객(%)"]).round(2)
    t.index = pd.MultiIndex.from_product([[item], t.index], names=["항목", "집단"])
    return t

t_age = compare("연령대", AGE_ORDER, "연령대")
t_gender = compare("성별", ["남", "여"], "gender")
t_prop = compare("성향", PROP_ORDER, "성향")
cmp = pd.concat([t_age, t_gender, t_prop])
print("비교 대상 — CoT 연령대", t_age["CoT 수"].sum(), "/ 성별", t_gender["CoT 수"].sum(), "/ 성향", t_prop["CoT 수"].sum(),
      "| 고객", t_age["고객 수"].sum())
cmp

In [ ]:
# 참고: 70대 + 80대 이상을 합친 '70대 이상'
x = t_age.droplevel(0).loc[["70대", "80대 이상"], ["CoT 수", "고객 수", "CoT(%)", "고객(%)"]].sum()
print(f"70대 이상 — CoT {int(x['CoT 수']):,}건 ({x['CoT(%)']:.1f}%) / 고객 {int(x['고객 수']):,}명 ({x['고객(%)']:.1f}%) / 차이 {x['CoT(%)'] - x['고객(%)']:+.1f}%p")

In [ ]:
# 참고: 분할별 CoT 비율 (Training / Validation)
for col, order in [("연령대", AGE_ORDER), ("gender", ["남", "여"]), ("성향", PROP_ORDER)]:
    t = pd.crosstab(sc.loc[sc[col].isin(order), col], sc.loc[sc[col].isin(order), "split"], normalize="columns").reindex(order)
    print((t * 100).round(1).to_string()); print()

## 4. 실제 고객 대비 CoT에서 많이 / 적게 다뤄진 집단 상위 3개

- 기준: 차이(%p). 항목(연령대·성별·성향)을 섞어 한 줄로 순위를 매긴다. 배율(CoT%/고객%)도 함께 적는다.

In [ ]:
rank = cmp.reset_index()[["항목", "집단", "CoT(%)", "고객(%)", "차이(%p)", "CoT/고객 배율"]]
print("[CoT에서 많이 다뤄진 상위 3]")
print(rank.sort_values("차이(%p)", ascending=False).head(3).to_string(index=False))
print()
print("[CoT에서 적게 다뤄진 상위 3]")
print(rank.sort_values("차이(%p)").head(3).to_string(index=False))

## 5. 그래프 5·6 → `outputs/figures/` (집계값만 사용)

In [ ]:
from src import plots
plots.setup()
import matplotlib.pyplot as plt

def compare_plot(t, name, title, xlabel):
    t = t.droplevel(0)
    fig, ax = plt.subplots(figsize=(9, 4.8))
    x = np.arange(len(t)); w = 0.38
    for i, (c, color, lab) in enumerate([("CoT(%)", "#2a6f97", f"CoT (증권 {t['CoT 수'].sum():,}건)"),
                                          ("고객(%)", "#f4a259", f"고객 DB (증권 고객 {t['고객 수'].sum():,}명)")]):
        bars = ax.bar(x + (i - 0.5) * w, t[c].values, w, label=lab, color=color)
        plots.bar_labels(ax, bars, "{:.1f}")
    ax.set_xticks(x, t.index); ax.set_xlabel(xlabel); ax.set_ylabel("비율 (%)"); ax.legend()
    return plots.save(fig, name, title)

print(compare_plot(t_age, "5_대표성_연령대", "대표성 비교: 연령대 분포 (CoT vs 고객 DB)", "연령대"))
print(compare_plot(t_prop, "6_대표성_성향", "대표성 비교: 투자성향 분포 (CoT vs 고객 DB)", "투자성향 (CoT는 D21 기준)"))

## 06 정리 (직접 작성)

위 결과를 보고 아래 질문에 대한 답을 적어 보세요. 작성 후 검토를 요청하면 됩니다.

1. 실제 고객 대비 CoT에서 많이·적게 다뤄진 집단은 무엇이고, 그 차이가 품질 진단 결과(결함 분포)를 해석할 때 어떤 한계가 되는가?
2. CoT의 연령·성별·성향 분포는 실제 고객 분포를 따르려 한 것으로 보이는가, 아니면 따로 설계된 분포로 보이는가? 그렇게 본 근거는?
3. 정제 데이터셋 리포트에서 '실제 고객층 대비 보강이 필요한 영역'을 어떤 집단·기준으로 제시할 것인가?

